In [1]:
import duckdb, time

con = duckdb.connect("../data/basket.duckdb")

tables = [
"aisles",
"departments",
"products",
"orders",
"order_products__prior",
'order_products__train',
]

for t in tables:
    t0 = time.time()
    con.execute(
        f"CREATE OR REPLACE TABLE {t} AS SELECT * FROM"
        f" read_csv_auto('../data/{t}.csv')"
    )
    n = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"{t:25s} {n:>12,} rows  ({time.time()-t0:.1f}s)")



aisles                             134 rows  (0.0s)
departments                         21 rows  (0.0s)
products                        49,688 rows  (0.0s)
orders                       3,421,083 rows  (1.4s)
order_products__prior       32,434,489 rows  (4.4s)
order_products__train        1,384,617 rows  (0.3s)


In [3]:
for t in tables:
  print(t)
  print(
      con.execute(f"DESCRIBE {t}")
      .df()[["column_name", "column_type"]]
      .to_string(index=False),
      "\n",
  )

aisles
column_name column_type
   aisle_id      BIGINT
      aisle     VARCHAR 

departments
  column_name column_type
department_id      BIGINT
   department     VARCHAR 

products
  column_name column_type
   product_id      BIGINT
 product_name     VARCHAR
     aisle_id      BIGINT
department_id      BIGINT 

orders
           column_name column_type
              order_id      BIGINT
               user_id      BIGINT
              eval_set     VARCHAR
          order_number      BIGINT
             order_dow      BIGINT
     order_hour_of_day     VARCHAR
days_since_prior_order      DOUBLE 

order_products__prior
      column_name column_type
         order_id      BIGINT
       product_id      BIGINT
add_to_cart_order      BIGINT
        reordered      BIGINT 

order_products__train
      column_name column_type
         order_id      BIGINT
       product_id      BIGINT
add_to_cart_order      BIGINT
        reordered      BIGINT 



In [7]:
con.sql("""
    SELECT eval_set, COUNT(*) AS orders, COUNT(DISTINCT user_id) AS users
    FROM orders 
    GROUP BY eval_set 
    ORDER BY orders DESC
""").df()

,eval_set,orders,users
0,prior,3214874,206209
1,train,131209,131209
2,test,75000,75000


In [8]:
con.sql("""
    SELECT  
      (SELECT COUNT(*) FROM order_products__prior op LEFT JOIN orders o USING(order_id) WHERE o.order_id IS NULL) AS prior_missing_order,
      (SELECT COUNT(*) FROM order_products__prior op LEFT JOIN products p USING(product_id) WHERE p.product_id IS NULL) AS prior_missing_product,
      (SELECT COUNT(*) FROM order_products__train op LEFT JOIN orders o USING(order_id) WHERE o.order_id IS NULL) AS train_missing_order
""").df()

,prior_missing_order,prior_missing_product,train_missing_order
0,0,0,0


In [ ]:
con.sql("""
    SELECT  
      SUM(CASE WHEN days_since_prior_order IS NULL AND order_number = 1 THEN 1 ELSE 0 END) AS null_on_first,
      SUM(CASE WHEN days_since_prior_order IS NULL AND order_number > 1 THEN 1 ELSE 0 END) AS null_elsewhere,
      MAX(days_since_prior_order) AS max_gap
    FROM orders
""").df()

,null_on_first,null_elsewhere,max_gap
0,206209.0,0.0,30.0


In [12]:
con.sql("""
SELECT COUNT(*) AS users_with_more_than_one_final
FROM (SELECT user_id FROM orders WHERE eval_set <> 'prior'
      GROUP BY user_id HAVING COUNT(*) > 1)
""").df()

,users_with_more_than_one_final
0,0


## Load summary
- 6 tables loaded into data/basket.duckdb; row counts match the source.
- eval_set: prior = history, train = last order (target), test = hidden.
- days_since_prior_order is capped at 30 → censored; null only on first orders.
- No orphan order lines or products.

In [15]:
import os; print(os.getcwd())

c:\Users\Joe\Projects\Basket\notebooks


In [16]:
con.close()